# Lecture 04: Classification & Information Theory (Fashion-MNIST)

**Machine Learning Applications in Physics (PHYG004)** — Sogang University, 2026 Spring

> **Data label**: Real data — Fashion-MNIST (Zalando Research, 70,000 28×28 grayscale images, 10 clothing classes).
> This is a non-physics canonical benchmark; physics connections (Ising phase classification) will appear in Lectures 06–07.

---

## Learning Objectives

1. Build and train a **linear regression** model from scratch in JAX; understand MSE as a log-likelihood
2. Understand the **normal equation** and its numerical solution via `jnp.linalg.solve`
3. Implement **full-batch gradient descent** with `jax.grad`; compare SGD optimizers using Optax
4. Apply **L2 regularization** (weight decay) and understand its Bayesian interpretation
5. Derive the **softmax function** and implement it in numerically stable form
6. Understand **entropy**, **cross-entropy**, and **KL divergence** via the physics analogy (Gibbs entropy)
7. Train softmax regression on **Fashion-MNIST** and interpret the confusion matrix
8. Connect cross-entropy minimization to **maximum likelihood estimation (MLE)**

---

### Session structure

| Section | Topic | Content |
|---------|-------|---------|
| §1–4 | Linear Regression | Model, MSE, normal equation, full-batch GD, Optax |
| §5 | L2 Regularization | Weight decay as Gaussian prior |
| §6 | Information Theory | Entropy, cross-entropy, KL divergence, physics analogies |
| §7 | Softmax Regression | Derivation, numerical stability, 2D demo |
| §8 | Fashion-MNIST | Load, explore, train, confusion matrix checkpoint |
| §9 | Exercises & Take-home | 3 in-session TODO checkpoints + 2 take-home extensions |


In [ ]:
# Install dependencies (run on Google Colab)
!pip install -q jax jaxlib optax flax matplotlib tensorflow tensorflow-datasets

In [ ]:
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
import numpy as np
import tensorflow_datasets as tfds

print(f"JAX version: {jax.__version__}")
print(f"Backend: {jax.default_backend()}")

key = jax.random.PRNGKey(42)

---
## §1. Linear Regression Model

The linear regression model predicts:

$$\hat{y} = Xw + b$$

where $X \in \mathbb{R}^{n \times d}$ is the input matrix ($n$ samples, $d$ features),
$w \in \mathbb{R}^d$ is the weight vector, and $b \in \mathbb{R}$ is the bias.

Equivalently, absorb the bias by augmenting $X$ with a column of ones:
$$\hat{y} = \tilde{X}\tilde{w}, \qquad \tilde{X} = [X \mid \mathbf{1}],\quad \tilde{w} = \begin{pmatrix} w \\ b \end{pmatrix}$$


In [ ]:
def linear_model(params: dict, X: jnp.ndarray) -> jnp.ndarray:
    """Linear regression: y_hat = X @ w + b."""
    return X @ params['w'] + params['b']

# Quick shape test
test_params = {'w': jnp.array([1.0, 2.0]), 'b': jnp.array(0.5)}
test_X = jnp.ones((3, 2))
y_hat = linear_model(test_params, test_X)
print(f"X shape: {test_X.shape}")
print(f"w shape: {test_params['w'].shape}")
print(f"y_hat shape: {y_hat.shape}")
print(f"y_hat = {y_hat}")

---
## §2. Synthetic Data Generation

We generate data from a known linear model with additive Gaussian noise:

$$y = Xw_{\text{true}} + b_{\text{true}} + \epsilon, \qquad \epsilon \sim \mathcal{N}(0, \sigma^2)$$


In [ ]:
# True parameters
w_true = jnp.array([2.0, -3.4])
b_true = 4.2          # non-zero mean important for §3 error-fix
n_samples = 1000
n_features = 2
noise_std = 0.5

# Generate data
key, k1, k2 = jax.random.split(key, 3)
X = jax.random.normal(k1, (n_samples, n_features))
noise = noise_std * jax.random.normal(k2, (n_samples,))
y = X @ w_true + b_true + noise

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"True weights: w = {w_true}, b = {b_true}")
print(f"Noise std: {noise_std}")

# Visualize: scatter plot of y vs each feature
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, ax in enumerate(axes):
    ax.scatter(X[:, i], y, s=5, alpha=0.3)
    ax.set_xlabel(f'$x_{i+1}$')
    ax.set_ylabel('$y$')
    ax.set_title(f'$y$ vs feature $x_{i+1}$ (true weight = {float(w_true[i]):.1f})')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## §3. Loss Function: Mean Squared Error

The MSE loss measures the average squared prediction error:

$$\mathcal{L}(w, b) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2 = \frac{1}{n}\|Xw + b - y\|^2$$

**Connection to maximum likelihood**: If $y_i = x_i^\top w + b + \epsilon_i$ with $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$,
then minimizing MSE is equivalent to maximizing the log-likelihood:

$$\log p(y \mid X, w, b) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i (y_i - x_i^\top w - b)^2$$

**Physics analogy**: MSE loss acts like a potential energy $U = \|\hat{y} - y\|^2 / n$.
Training is energy minimization; the analytic solution is the ground state.

> **What is MSE at zero parameters?**
> Setting $w=0, b=0$ gives $\hat{y} = 0$, so $\text{MSE}(0) = \frac{1}{n}\sum y_i^2 = E[y^2]$.
> By the variance decomposition: $E[y^2] = \text{Var}(y) + (\text{mean}(y))^2$.
> Here $b_{\text{true}} = 4.2$, so $\text{mean}(y) \approx 4.2$ and $(\text{mean}(y))^2 \approx 17.6$ **dominates**.
> This decomposition is the first preview of bias–variance: the $\text{mean}(y)^2$ term is pure **bias** (our zero-model is systematically wrong), and $\text{Var}(y)$ is **variance** (spread of the data). We will revisit this in Lecture 05.


In [ ]:
def mse_loss(params: dict, X: jnp.ndarray, y: jnp.ndarray) -> jnp.ndarray:
    """Mean squared error loss."""
    y_hat = linear_model(params, X)
    return jnp.mean((y_hat - y) ** 2)

# Test with zero params — §4 error fix #17
init_params = {'w': jnp.zeros(n_features), 'b': jnp.array(0.0)}
loss_val = mse_loss(init_params, X, y)

# Correct decomposition: E[y²] = Var(y) + mean(y)²
var_y = jnp.var(y)
mean_sq = jnp.mean(y) ** 2
print(f"Loss with zero params (w=0, b=0): {loss_val:.4f}")
print(f"  = E[y²] = Var(y) + mean(y)²")
print(f"  = {var_y:.4f} + {mean_sq:.4f}")
print(f"  = {float(var_y + mean_sq):.4f}  ✓ matches loss")
print()
print(f"  Var(y) ≈ noise variance σ² = {noise_std**2:.4f} (plus small contrib from X)")
print(f"  mean(y)² ≈ b_true² = {b_true**2:.4f}  ← dominant term (bias of the zero model)")
print()
print(f"  → MSE(0) ≈ {float(loss_val):.4f}, NOT Var(y) = {float(var_y):.4f}")
print(f"  The difference is entirely the squared mean: b_true² ≈ {b_true**2:.2f}")
print(f"  Forward-pointer: L05 MLP deep-dive will formalize bias–variance decomposition.")

---
## §4. Analytic Solution: Normal Equation

Setting $\nabla_w \mathcal{L} = 0$ gives the **normal equation**:

$$\tilde{w}^* = (\tilde{X}^\top \tilde{X})^{-1} \tilde{X}^\top y$$

where $\tilde{X} = [X \mid \mathbf{1}]$ includes the bias column.
We use `jnp.linalg.solve` instead of computing the inverse directly — more numerically stable.


In [ ]:
# Augment X with a column of ones for the bias
X_aug = jnp.concatenate([X, jnp.ones((n_samples, 1))], axis=1)
print(f"X_aug shape: {X_aug.shape}")

# Solve normal equation: (X^T X) w = X^T y
w_analytic = jnp.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)

w_solved = w_analytic[:n_features]
b_solved = w_analytic[n_features]

print(f"\nAnalytic solution:")
print(f"  w = {w_solved}")
print(f"  b = {b_solved:.4f}")
print(f"\nTrue parameters:")
print(f"  w = {w_true}")
print(f"  b = {b_true}")
print(f"\nRecovery error:")
print(f"  |w - w_true| = {jnp.linalg.norm(w_solved - w_true):.6f}")
print(f"  |b - b_true| = {abs(float(b_solved) - b_true):.6f}")

analytic_params = {'w': w_solved, 'b': b_solved}
analytic_loss = mse_loss(analytic_params, X, y)
print(f"\nMSE at analytic solution: {analytic_loss:.6f}")
print(f"(Should be close to noise variance = {noise_std**2:.4f})")

---
## §5. Full-Batch Gradient Descent

Gradient descent updates parameters iteratively:

$$w \leftarrow w - \eta \nabla_w \mathcal{L}, \qquad b \leftarrow b - \eta \nabla_b \mathcal{L}$$

We use `jax.grad` to compute gradients automatically.
**Full-batch** gradient descent uses all data at every step — guaranteed descent for convex problems.


In [ ]:
grad_fn = jax.grad(mse_loss)

params = {'w': jnp.zeros(n_features), 'b': jnp.array(0.0)}
lr = 0.1
n_epochs = 100

losses_gd = []
params_history = []

for epoch in range(n_epochs):
    loss = mse_loss(params, X, y)
    losses_gd.append(float(loss))
    grads = grad_fn(params, X, y)
    params = {
        'w': params['w'] - lr * grads['w'],
        'b': params['b'] - lr * grads['b'],
    }
    if epoch % 20 == 0:
        params_history.append((epoch, params['w'].copy(), params['b']))

print(f"Learned: w = {params['w']}, b = {params['b']:.4f}")
print(f"True:    w = {w_true}, b = {b_true}")
print(f"Final loss: {losses_gd[-1]:.6f}")
print(f"Analytic loss: {float(analytic_loss):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses_gd, 'b-', lw=1.5)
axes[0].axhline(float(analytic_loss), color='r', ls='--', lw=1, label='Analytic optimum')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Full-Batch Gradient Descent')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

epochs_h = [e for e, _, _ in params_history]
w0_h = [float(w[0]) for _, w, _ in params_history]
w1_h = [float(w[1]) for _, w, _ in params_history]
b_h  = [float(b) for _, _, b in params_history]
axes[1].plot(epochs_h, w0_h, 'o-', label=f'$w_1$ (true={float(w_true[0]):.1f})')
axes[1].plot(epochs_h, w1_h, 's-', label=f'$w_2$ (true={float(w_true[1]):.1f})')
axes[1].plot(epochs_h, b_h,  '^-', label=f'$b$ (true={b_true:.1f})')
for val, color in [(float(w_true[0]),'C0'), (float(w_true[1]),'C1'), (b_true,'C2')]:
    axes[1].axhline(val, color=color, ls=':', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Parameter value')
axes[1].set_title('Parameter Convergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## §6. Concise Implementation with Optax

Optax provides standard optimizers with a clean pattern:
1. `tx = optax.adam(lr)` — create optimizer
2. `opt_state = tx.init(params)` — initialize state
3. Each step: compute grads → `updates, opt_state = tx.update(grads, opt_state, params)` → `params = optax.apply_updates(params, updates)`


In [ ]:
def train_optax(X, y, optimizer, n_epochs):
    """Train linear regression with an Optax optimizer."""
    params = {'w': jnp.zeros(X.shape[1]), 'b': jnp.array(0.0)}
    opt_state = optimizer.init(params)

    @jax.jit
    def step(params, opt_state):
        loss, grads = jax.value_and_grad(mse_loss)(params, X, y)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, loss

    losses = []
    for epoch in range(n_epochs):
        params, opt_state, loss = step(params, opt_state)
        losses.append(float(loss))
    return params, losses

n_epochs = 200
optimizers = {
    'SGD (lr=0.1)': optax.sgd(0.1),
    'SGD + momentum (lr=0.1)': optax.sgd(0.1, momentum=0.9),
    'Adam (lr=0.01)': optax.adam(0.01),
    'Adam (lr=0.1)': optax.adam(0.1),
}

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for name, opt in optimizers.items():
    _, losses_opt = train_optax(X, y, opt, n_epochs)
    ax.plot(losses_opt, lw=1.5, label=name)
ax.axhline(float(analytic_loss), color='k', ls='--', lw=1, label='Optimum')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Optimizer Comparison on Linear Regression')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final parameters for Adam
params_adam, _ = train_optax(X, y, optax.adam(0.1), n_epochs)
print(f"Adam result: w = {params_adam['w']}, b = {params_adam['b']:.4f}")
print(f"True:        w = {w_true}, b = {b_true}")

---
## §7. L2 Regularization (Weight Decay)

When a model has too many parameters relative to the data, it can **overfit** by fitting noise.
**Weight decay** adds a penalty on the weight magnitude:

$$\mathcal{L}_{\text{reg}}(w, b) = \frac{1}{n}\|Xw + b - y\|^2 + \lambda \|w\|^2$$

**Bayesian interpretation**: Weight decay is equivalent to a Gaussian prior $p(w) \propto \exp(-\lambda\|w\|^2)$
— MAP estimation rather than MLE.

> **Physics analogy**: $\lambda\|w\|^2$ is an elastic restoring term (spring potential) that prevents
> the weights from wandering far from zero. The competition between data fit and regularization
> is exactly the competition between energy and entropy in statistical mechanics.

**Demo**: We fit a degree-14 polynomial (14 parameters) to only 15 noisy data points.
Without regularization the polynomial oscillates wildly; weight decay forces a smooth fit.

> **Take-home Extension A** (moved from this session): After working through this demo,
> perform a full sweep of regularization strengths (λ from 0 to 5) and plot the bias–variance
> trade-off. Bring your plot to Lecture 05 where we will unify it with the MLP overfitting demo.


In [ ]:
# --- Polynomial regression overfitting demo ---
poly_degree = 14
n_train_poly = 15
n_test_poly = 200

key, k1, k2, k3 = jax.random.split(key, 4)

y_true_fn = lambda x: jnp.sin(2 * jnp.pi * x)
x_train_poly = jnp.sort(jax.random.uniform(k1, (n_train_poly,)))
y_train_poly = y_true_fn(x_train_poly) + 0.3 * jax.random.normal(k2, (n_train_poly,))
x_test_poly = jnp.linspace(0, 1, n_test_poly)
y_test_poly = y_true_fn(x_test_poly) + 0.3 * jax.random.normal(k3, (n_test_poly,))

def poly_features(x, degree):
    return jnp.stack([x ** i for i in range(1, degree + 1)], axis=1)

X_train_poly = poly_features(x_train_poly, poly_degree)
X_test_poly  = poly_features(x_test_poly, poly_degree)
poly_mean = X_train_poly.mean(axis=0)
poly_std  = X_train_poly.std(axis=0) + 1e-8
X_train_poly = (X_train_poly - poly_mean) / poly_std
X_test_poly  = (X_test_poly  - poly_mean) / poly_std

print(f"Poly degree: {poly_degree}, n_train={n_train_poly}, n/d={n_train_poly/poly_degree:.2f}")

def train_with_wd(X_tr, y_tr, X_te, y_te, optimizer, n_epochs=2000, weight_decay=0.0):
    params = {'w': jnp.zeros(X_tr.shape[1]), 'b': jnp.array(0.0)}
    opt_state = optimizer.init(params)

    def loss_fn(params):
        return mse_loss(params, X_tr, y_tr) + weight_decay * jnp.sum(params['w'] ** 2)

    @jax.jit
    def step(params, opt_state):
        _, grads = jax.value_and_grad(loss_fn)(params)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state

    tr_losses, te_losses = [], []
    for _ in range(n_epochs):
        params, opt_state = step(params, opt_state)
        tr_losses.append(float(mse_loss(params, X_tr, y_tr)))
        te_losses.append(float(mse_loss(params, X_te, y_te)))
    return params, tr_losses, te_losses

params_no_wd,  _, te_no_wd = train_with_wd(X_train_poly, y_train_poly, X_test_poly, y_test_poly, optax.adam(1e-2))
params_wd,     _, te_wd    = train_with_wd(X_train_poly, y_train_poly, X_test_poly, y_test_poly, optax.adam(1e-2), weight_decay=0.01)
params_wd_str, _, te_wd_s  = train_with_wd(X_train_poly, y_train_poly, X_test_poly, y_test_poly, optax.adam(1e-2), weight_decay=0.1)

x_plot = jnp.linspace(0, 1, 500)
X_plot_poly = (poly_features(x_plot, poly_degree) - poly_mean) / poly_std
y_no_wd = linear_model(params_no_wd, X_plot_poly)
y_wd    = linear_model(params_wd, X_plot_poly)
y_wd_s  = linear_model(params_wd_str, X_plot_poly)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.scatter(x_train_poly, y_train_poly, c='k', s=40, zorder=5, label='Training data')
ax.plot(x_plot, y_true_fn(x_plot), 'k--', lw=1.5, label='True sin(2πx)')
ax.plot(x_plot, y_no_wd, 'C0-', lw=2, label='No WD')
ax.plot(x_plot, y_wd,    'C3-', lw=2, label='WD λ=0.01')
ax.plot(x_plot, y_wd_s,  'C2-', lw=2, label='WD λ=0.1')
ax.set_ylim(-2, 2)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('Polynomial Fit (degree 14, 15 samples)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(range(poly_degree), np.abs(np.array(params_no_wd['w'])), alpha=0.7, label='No WD', color='C0')
ax.bar(range(poly_degree), np.abs(np.array(params_wd['w'])),    alpha=0.7, label='WD 0.01', color='C3')
ax.bar(range(poly_degree), np.abs(np.array(params_wd_str['w'])),alpha=0.7, label='WD 0.1',  color='C2')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('|w_i|')
ax.set_title('Weight Magnitudes')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Final test MSE:")
print(f"  No weight decay:     {te_no_wd[-1]:.4f}")
print(f"  WD λ=0.01:           {te_wd[-1]:.4f}")
print(f"  WD λ=0.1:            {te_wd_s[-1]:.4f}")

---
## §8. Information Theory: Entropy, Cross-Entropy, KL Divergence

### Entropy — "How surprised are you on average?"

The **entropy** of a distribution $\mathbf{p}$ over $K$ outcomes:

$$H(\mathbf{p}) = -\sum_k p_k \log p_k$$

- Uniform distribution over $K$ classes: maximum entropy $H = \log K$
- One-hot (certainty): minimum entropy $H = 0$

> **Physics analogy — Gibbs entropy**:
> In statistical mechanics, $S = -k_B \sum_i p_i \ln p_i$ is the Gibbs entropy.
> Maximizing it (subject to constraints like fixed $\langle E \rangle$) gives the Boltzmann distribution.
> Information-theoretic entropy $H$ is the same formula up to $k_B$ — it measures the "missing
> information" about the microstate.

### Cross-Entropy — "How surprised if you use the wrong model?"

The **cross-entropy** between true distribution $\mathbf{y}$ and predicted $\hat{\mathbf{y}}$:

$$H(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{k=1}^{K} y_k \log \hat{y}_k$$

For one-hot labels ($y_c = 1$, others $0$): simplifies to $\ell = -\log \hat{y}_c$ — the NLL of the correct class.

### KL Divergence — "Extra surprise from using the wrong model"

$$D_{\text{KL}}(\mathbf{y} \| \hat{\mathbf{y}}) = H(\mathbf{y}, \hat{\mathbf{y}}) - H(\mathbf{y}) = \sum_k y_k \log \frac{y_k}{\hat{y}_k}$$

Properties: $D_{\text{KL}} \geq 0$ (equality iff $\hat{\mathbf{y}} = \mathbf{y}$); **not symmetric**.

> **KL divergence as a free energy difference**:
> Suppose $\mathbf{y}$ is the true Boltzmann distribution $p^* \propto e^{-\beta E}$ and
> $\hat{\mathbf{y}}$ is your model's approximate distribution $q$.
> Then $D_{\text{KL}}(p^* \| q) = \beta(F_q - F^*)$ where $F = \langle E \rangle - TS$ is the free energy.
> **Minimizing KL = minimizing variational free energy under the wrong model.**
> (This connection deepens in Lecture 14 with VAEs, where the ELBO is a variational free energy.)

### Why minimize cross-entropy ≡ minimize KL

Since $H(\mathbf{y})$ is constant (determined by the labels, equals 0 for one-hot):

$$\arg\min_{\hat{\mathbf{y}}} H(\mathbf{y}, \hat{\mathbf{y}}) = \arg\min_{\hat{\mathbf{y}}} D_{\text{KL}}(\mathbf{y} \| \hat{\mathbf{y}})$$

| Quantity | Formula | Physics meaning |
|----------|---------|-----------------|
| $H(\mathbf{y})$ | $-\sum_k y_k \log y_k$ | Gibbs entropy of data labels |
| $H(\mathbf{y}, \hat{\mathbf{y}})$ | $-\sum_k y_k \log \hat{y}_k$ | Avg. surprise using model $\hat{\mathbf{y}}$ |
| $D_{\text{KL}}(\mathbf{y}\|\hat{\mathbf{y}})$ | $H(\mathbf{y},\hat{\mathbf{y}}) - H(\mathbf{y})$ | Extra free energy from wrong model |


In [ ]:
# Numerical demonstration of entropy, cross-entropy, KL divergence

# Entropy of several distributions
def entropy(p, eps=1e-12):
    """H(p) = -sum p_k log p_k"""
    return -jnp.sum(p * jnp.log(p + eps))

def cross_entropy_dist(p, q, eps=1e-12):
    """H(p, q) = -sum p_k log q_k"""
    return -jnp.sum(p * jnp.log(q + eps))

def kl_divergence(p, q, eps=1e-12):
    """KL(p || q) = sum p_k log(p_k / q_k)"""
    return jnp.sum(p * jnp.log((p + eps) / (q + eps)))

K = 5
uniform = jnp.ones(K) / K
peaked  = jnp.array([0.8, 0.05, 0.05, 0.05, 0.05])
one_hot = jnp.array([1.0, 0.0, 0.0, 0.0, 0.0])

print("Entropy H(p):")
print(f"  Uniform ({K} classes): H = {entropy(uniform):.4f}  (max = log {K} = {jnp.log(K):.4f})")
print(f"  Peaked:                H = {entropy(peaked):.4f}")
print(f"  One-hot (certain):     H = {entropy(one_hot):.4f}  (min = 0)")

print("\nKL Divergence KL(p_true || p_wrong):")
print(f"  KL(peaked || uniform)  = {kl_divergence(peaked, uniform):.4f}")
print(f"  KL(uniform || peaked)  = {kl_divergence(uniform, peaked):.4f}  (not symmetric!)")
print(f"  KL(peaked || peaked)   = {kl_divergence(peaked, peaked):.4f}  (zero)")

# Verify: CE = H(p) + KL(p||q)
print("\nVerify H(p,q) = H(p) + KL(p||q):")
p, q = peaked, uniform
print(f"  H(p,q)         = {cross_entropy_dist(p, q):.4f}")
print(f"  H(p) + KL(p||q) = {entropy(p):.4f} + {kl_divergence(p,q):.4f} = {entropy(p)+kl_divergence(p,q):.4f}")

# Visualize entropy landscape
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: binary entropy vs p
p_vals = jnp.linspace(0.01, 0.99, 200)
h_binary = -(p_vals * jnp.log(p_vals) + (1-p_vals) * jnp.log(1-p_vals))
axes[0].plot(p_vals, h_binary, 'b-', lw=2)
axes[0].axvline(0.5, color='r', ls='--', alpha=0.6, label='Maximum (p=0.5)')
axes[0].set_xlabel('p (probability of class 1)')
axes[0].set_ylabel('Entropy H(p, 1-p)')
axes[0].set_title('Binary Entropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: KL(peaked || q) as q varies for 2-class case
p_fixed = jnp.array([0.8, 0.2])
q1_vals = jnp.linspace(0.01, 0.99, 200)
kl_vals = jnp.array([kl_divergence(p_fixed, jnp.array([q1, 1-q1])) for q1 in q1_vals])
axes[1].plot(q1_vals, kl_vals, 'C1-', lw=2)
axes[1].axvline(0.8, color='r', ls='--', alpha=0.6, label='Minimum at q=p (KL=0)')
axes[1].set_xlabel('q₁ (predicted prob. for class 1)')
axes[1].set_ylabel('KL(p || q)')
axes[1].set_title('KL Divergence (p=[0.8,0.2] fixed)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## §9. Softmax Function

Given a vector of logits $\mathbf{o} \in \mathbb{R}^K$, the **softmax** produces a probability distribution:

$$\hat{y}_k = \text{softmax}(\mathbf{o})_k = \frac{\exp(o_k)}{\sum_{j=1}^{K} \exp(o_j)}$$

**Numerical issue**: if any $o_k$ is large, $\exp(o_k)$ overflows to `inf` → `nan`.
The **log-sum-exp trick** subtracts the maximum before exponentiating:

$$\text{softmax}(\mathbf{o})_k = \frac{\exp(o_k - \max(\mathbf{o}))}{\sum_j \exp(o_j - \max(\mathbf{o}))}$$

---
### Exercise 1 — Softmax numerical stability TODO

Implement both versions, then verify the following checkpoints.


In [ ]:
# Exercise 1: Softmax numerical stability

def softmax_naive(logits):
    """Naive softmax: numerically unstable for large logits."""
    exps = jnp.exp(logits)
    return exps / jnp.sum(exps, axis=-1, keepdims=True)

# TODO: Implement the numerically stable version using the log-sum-exp trick.
def softmax_stable(logits):
    """Numerically stable softmax.

    Steps:
      1. Compute m = max(logits)
      2. Compute shifted = logits - m
      3. Return exp(shifted) / sum(exp(shifted))
    """
    # YOUR CODE HERE
    raise NotImplementedError

# ─── Checkpoint 1a: both agree on small logits ────────────────────────────
o_small = jnp.array([2.0, 1.0, 0.1])
# assert jnp.allclose(softmax_naive(o_small), softmax_stable(o_small), atol=1e-5), "Checkpoint 1a failed"
# print("Checkpoint 1a passed: small logits agree")

# ─── Checkpoint 1b: naive overflows on large logits ───────────────────────
o_large = jnp.array([1000.0, 1001.0, 1002.0])
# assert jnp.any(jnp.isnan(softmax_naive(o_large))), "Checkpoint 1b: expected NaN from naive"
# assert not jnp.any(jnp.isnan(softmax_stable(o_large))), "Checkpoint 1b: stable should not NaN"
# print("Checkpoint 1b passed: large logits handled correctly")

# Reference: JAX built-in (always numerically stable)
print("Large logits — JAX reference:", jax.nn.softmax(o_large))
print("Expected: [~0.09, ~0.24, ~0.67]")

---
## §10. Cross-Entropy Loss from Scratch

---
### Exercise 2 — Cross-entropy = MLE derivation (markdown TODO)

The multinomial likelihood for $N$ samples with true one-hot labels $y_{ik}$ and predicted probabilities $\hat{p}_{ik}$:

$$L = \prod_{i=1}^{N} \prod_{k=1}^{K} \hat{p}_{ik}^{y_{ik}}$$

**TODO**: Derive that $-\log L = \sum_{i,k} y_{ik} \log \hat{p}_{ik}$ is exactly the cross-entropy sum.

> **Steps**:
> 1. Take $-\log$ of $L$: pull the product into a sum.
> 2. Use the one-hot constraint: for each sample $i$, only the true class $c_i$ contributes.
> 3. Show that $-\log L = -\sum_i \log \hat{p}_{i,c_i}$ = cross-entropy.
> 4. Conclude: **minimizing cross-entropy = maximizing the multinomial log-likelihood** (MLE).

*(Write your derivation in the cell below or in your notes.)*


In [ ]:
# Cross-entropy loss implementation
def cross_entropy_loss(logits: jnp.ndarray, labels: jnp.ndarray) -> jnp.ndarray:
    """Cross-entropy loss from logits and integer labels.

    Args:
        logits: (N, K) raw scores (pre-softmax)
        labels: (N,) integer class labels in {0, ..., K-1}
    Returns:
        scalar mean loss
    """
    # Use log-softmax for numerical stability: log(softmax(o)) computed jointly
    log_probs = jax.nn.log_softmax(logits)   # (N, K)
    # Pick the log-prob of the correct class for each sample
    nll = -log_probs[jnp.arange(labels.shape[0]), labels]  # (N,)
    return jnp.mean(nll)

# Verify against optax reference
key, k1 = jax.random.split(key)
logits_test = jax.random.normal(k1, (4, 3))
labels_test = jnp.array([0, 2, 1, 0])
one_hot_test = jax.nn.one_hot(labels_test, 3)

loss_ours  = cross_entropy_loss(logits_test, labels_test)
loss_optax = jnp.mean(optax.softmax_cross_entropy(logits_test, one_hot_test))

print(f"Our cross-entropy:   {loss_ours:.6f}")
print(f"Optax cross-entropy: {loss_optax:.6f}")
print(f"Match: {jnp.allclose(loss_ours, loss_optax)}")
print(f"Logits shape: {logits_test.shape}, Labels shape: {labels_test.shape}")

---
## §11. Softmax Regression on Synthetic 2D Data

Before jumping to Fashion-MNIST, we visualize softmax regression on a simple 2D dataset (3 linearly separable Gaussian clouds), so we can plot the decision boundaries.

Model: $\mathbf{o} = X W + \mathbf{b}$, then $\hat{\mathbf{y}} = \text{softmax}(\mathbf{o})$,
with $W \in \mathbb{R}^{d \times K}$, $\mathbf{b} \in \mathbb{R}^K$.


In [ ]:
# Generate 2D data with 3 linearly separable classes
n_per_class = 100
n_classes = 3
means = jnp.array([[2.0, 0.0], [-1.0, 1.7], [-1.0, -1.7]])
cov = jnp.eye(2) * 0.5

X_list, y_list = [], []
for c in range(n_classes):
    key, k1 = jax.random.split(key)
    X_c = jax.random.multivariate_normal(k1, means[c], cov, (n_per_class,))
    X_list.append(X_c)
    y_list.append(jnp.full(n_per_class, c))

X_synth = jnp.concatenate(X_list)
y_synth = jnp.concatenate(y_list)

key, k1 = jax.random.split(key)
perm = jax.random.permutation(k1, X_synth.shape[0])
X_synth, y_synth = X_synth[perm], y_synth[perm]

print(f"X_synth shape: {X_synth.shape}")
print(f"y_synth shape: {y_synth.shape}")

colors = ['#e41a1c', '#377eb8', '#4daf4a']
fig, ax = plt.subplots(figsize=(6, 5))
for c in range(n_classes):
    mask = y_synth == c
    ax.scatter(X_synth[mask, 0], X_synth[mask, 1],
               c=colors[c], label=f'Class {c}', alpha=0.6, s=30, edgecolors='w', linewidth=0.5)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Synthetic 2D Classification (3 classes)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Softmax regression from scratch on 2D data
n_feat_synth = 2

key, k1, k2 = jax.random.split(key, 3)
W = jax.random.normal(k1, (n_feat_synth, n_classes)) * 0.01
b = jnp.zeros(n_classes)
params_synth = (W, b)

def model_synth(params, X):
    """Softmax regression: returns logits."""
    W, b = params
    return X @ W + b  # (N, K)

def loss_synth(params, X, y):
    return cross_entropy_loss(model_synth(params, X), y)

def accuracy_fn(params, X, y, model_fn):
    preds = jnp.argmax(model_fn(params, X), axis=-1)
    return jnp.mean(preds == y)

grad_fn_synth = jax.jit(jax.grad(loss_synth))
lr = 0.1
loss_history = []

for epoch in range(200):
    grads = grad_fn_synth(params_synth, X_synth, y_synth)
    W, b = params_synth
    params_synth = (W - lr * grads[0], b - lr * grads[1])
    if epoch % 20 == 0:
        loss_history.append(float(loss_synth(params_synth, X_synth, y_synth)))

final_acc = float(accuracy_fn(params_synth, X_synth, y_synth, model_synth))
print(f"Final accuracy: {final_acc:.4f}")

# Plot loss curve and decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(range(0, 200, 20), loss_history, 'b-o', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss (2D Synthetic)')
axes[0].grid(True, alpha=0.3)

ax = axes[1]
x_min, x_max = X_synth[:, 0].min() - 1, X_synth[:, 0].max() + 1
y_min, y_max = X_synth[:, 1].min() - 1, X_synth[:, 1].max() + 1
xx, yy = jnp.meshgrid(jnp.linspace(x_min, x_max, 200), jnp.linspace(y_min, y_max, 200))
grid_pts = jnp.column_stack([xx.ravel(), yy.ravel()])
preds_grid = jnp.argmax(model_synth(params_synth, grid_pts), axis=-1).reshape(xx.shape)
ax.contourf(xx, yy, preds_grid, levels=[-0.5, 0.5, 1.5, 2.5],
            colors=['#fbb4ae', '#b3cde3', '#ccebc5'], alpha=0.5)
for c in range(n_classes):
    mask = y_synth == c
    ax.scatter(X_synth[mask, 0], X_synth[mask, 1],
               c=colors[c], label=f'Class {c}', alpha=0.7, s=30, edgecolors='w', linewidth=0.5)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Decision Boundaries (linear — softmax regression)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## §12. Fashion-MNIST Dataset

**Real data**: Fashion-MNIST (Zalando Research).
70,000 grayscale images, 28×28 pixels, 10 clothing classes.
A direct drop-in replacement for MNIST — harder (no simple stroke patterns), yet still tractable for softmax regression.

| Label | Class | Label | Class |
|-------|-------|-------|-------|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

**Physics connection** (forward pointer): Ising MC spin configurations will appear as
a similar 28×28 binary image classification task in Lectures 06–07, where a CNN will
dramatically outperform this linear softmax baseline.


In [ ]:
# Load Fashion-MNIST via tensorflow_datasets
ds_train, ds_test = tfds.load(
    'fashion_mnist',
    split=['train', 'test'],
    as_supervised=True,
    batch_size=-1,
)

X_train_img = jnp.array(ds_train[0], dtype=jnp.float32) / 255.0
y_train = jnp.array(ds_train[1])
X_test_img  = jnp.array(ds_test[0],  dtype=jnp.float32) / 255.0
y_test  = jnp.array(ds_test[1])

print(f"Training set:  X={X_train_img.shape}, y={y_train.shape}")
print(f"Test set:      X={X_test_img.shape},  y={y_test.shape}")
print(f"Pixel range:   [{float(X_train_img.min()):.1f}, {float(X_train_img.max()):.1f}]")

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train_img[i].squeeze(), cmap='gray_r')
    ax.set_title(class_names[int(y_train[i])], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST Samples', fontsize=14)
plt.tight_layout()
plt.show()

---
## §13. Softmax Regression on Fashion-MNIST (from Scratch)

We flatten each 28×28 image to a 784-d vector and apply softmax regression:
$W \in \mathbb{R}^{784 \times 10}$, $\mathbf{b} \in \mathbb{R}^{10}$ — total 7,850 parameters.

**Minibatch SGD**: at each epoch, shuffle the data and process batches of 256 samples.


In [ ]:
# Flatten: (N, 28, 28, 1) -> (N, 784)
X_train_flat = X_train_img.reshape(X_train_img.shape[0], -1)
X_test_flat  = X_test_img.reshape(X_test_img.shape[0], -1)

X_val = X_train_flat[50000:]
y_val = y_train[50000:]
X_tr  = X_train_flat[:50000]
y_tr  = y_train[:50000]

print(f"X_tr:   {X_tr.shape},   y_tr:  {y_tr.shape}")
print(f"X_val:  {X_val.shape},  y_val: {y_val.shape}")
print(f"X_test: {X_test_flat.shape}, y_test: {y_test.shape}")

key, k1 = jax.random.split(key)
W_mnist = jax.random.normal(k1, (784, 10)) * 0.01
b_mnist = jnp.zeros(10)
params_mnist = (W_mnist, b_mnist)
print(f"\nParameters: W={W_mnist.shape}, b={b_mnist.shape}, total={W_mnist.size + b_mnist.size:,}")

def loss_mnist(params, X, y):
    W, b = params
    logits = X @ W + b
    return cross_entropy_loss(logits, y)

def acc_mnist(params, X, y):
    W, b = params
    preds = jnp.argmax(X @ W + b, axis=-1)
    return jnp.mean(preds == y)

@jax.jit
def train_step_mnist(params, X_batch, y_batch, lr=0.1):
    grads = jax.grad(loss_mnist)(params, X_batch, y_batch)
    W, b = params
    gW, gb = grads
    return (W - lr * gW, b - lr * gb)

batch_size = 256
n_epochs_mnist = 10
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(n_epochs_mnist):
    key, k1 = jax.random.split(key)
    perm = jax.random.permutation(k1, X_tr.shape[0])
    X_shuf, y_shuf = X_tr[perm], y_tr[perm]

    for i in range(X_tr.shape[0] // batch_size):
        Xb = X_shuf[i*batch_size:(i+1)*batch_size]
        yb = y_shuf[i*batch_size:(i+1)*batch_size]
        params_mnist = train_step_mnist(params_mnist, Xb, yb)

    tr_loss  = float(loss_mnist(params_mnist, X_tr, y_tr))
    val_loss = float(loss_mnist(params_mnist, X_val, y_val))
    val_acc  = float(acc_mnist(params_mnist, X_val, y_val))
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1:2d}/{n_epochs_mnist} | "
          f"Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

test_acc = float(acc_mnist(params_mnist, X_test_flat, y_test))
print(f"\nTest accuracy: {test_acc:.4f}  (expected ~0.83-0.85)")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ep_range = range(1, n_epochs_mnist + 1)

axes[0].plot(ep_range, history['train_loss'], 'b-o', label='Train', markersize=4)
axes[0].plot(ep_range, history['val_loss'],   'r-s', label='Validation', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Loss (Fashion-MNIST)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, history['val_acc'], 'g-o', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Softmax Regression on Fashion-MNIST (from scratch)', fontsize=13)
plt.tight_layout()
plt.show()

# Show some predictions
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flatten()):
    W, b = params_mnist
    pred = int(jnp.argmax(X_test_flat[i] @ W + b))
    true = int(y_test[i])
    color = 'green' if pred == true else 'red'
    ax.imshow(X_test_img[i].squeeze(), cmap='gray_r')
    ax.set_title(class_names[pred], fontsize=8, color=color)
    ax.axis('off')
plt.suptitle('Test Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

---
## §14. Confusion Matrix

A confusion matrix shows, for each true class (rows), how often the model predicted each other class (columns).
Off-diagonal entries reveal **which classes are most easily confused**.

---
### Exercise 3 — Confusion matrix interpretation checkpoint

After running the cell below, answer in your notes:

1. Which two classes are most frequently confused? (Look for the largest off-diagonal entry.)
2. Is this physically/visually reasonable? (What do the garment textures/shapes have in common?)
3. Why does a **linear** softmax model struggle to separate them — what additional feature would help?

> **Expected finding**: Pullover (2) and Shirt (6) are typically the most confused pair on Fashion-MNIST.
> Both are upper-body garments with similar overall pixel distributions; distinguishing them requires
> finer texture or structural information that a linear model cannot extract from raw pixels alone.
> A CNN with learnable local filters (Lecture 06) will resolve this significantly.


In [ ]:
# Exercise 3: Confusion matrix checkpoint
W, b = params_mnist
test_logits = X_test_flat @ W + b
test_preds  = jnp.argmax(test_logits, axis=-1)

# Build confusion matrix manually
n_cls = 10
conf_mat = np.zeros((n_cls, n_cls), dtype=int)
for true, pred in zip(np.array(y_test), np.array(test_preds)):
    conf_mat[true, pred] += 1

# Normalize row-wise (recall per true class)
conf_mat_norm = conf_mat.astype(float) / conf_mat.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
im0 = axes[0].imshow(conf_mat, cmap='Blues')
axes[0].set_xticks(range(n_cls))
axes[0].set_yticks(range(n_cls))
axes[0].set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
axes[0].set_yticklabels(class_names, fontsize=8)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (counts)')
for i in range(n_cls):
    for j in range(n_cls):
        axes[0].text(j, i, str(conf_mat[i, j]), ha='center', va='center',
                     fontsize=6, color='white' if conf_mat[i,j] > 400 else 'black')
plt.colorbar(im0, ax=axes[0])

# Normalized (row = recall)
im1 = axes[1].imshow(conf_mat_norm, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_xticks(range(n_cls))
axes[1].set_yticks(range(n_cls))
axes[1].set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
axes[1].set_yticklabels(class_names, fontsize=8)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Confusion Matrix (normalized by true class)')
for i in range(n_cls):
    for j in range(n_cls):
        axes[1].text(j, i, f'{conf_mat_norm[i,j]:.2f}', ha='center', va='center',
                     fontsize=6, color='black')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Fashion-MNIST Softmax Regression — Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

# ─── Checkpoint 3: find the most confused pair ───────────────────────────
off_diag = conf_mat.copy()
np.fill_diagonal(off_diag, 0)
most_confused_idx = np.unravel_index(np.argmax(off_diag), off_diag.shape)
true_cls, pred_cls = most_confused_idx
print(f"\nMost confused pair:")
print(f"  True: {class_names[true_cls]} ({true_cls})  →  Predicted: {class_names[pred_cls]} ({pred_cls})")
print(f"  Count: {off_diag[true_cls, pred_cls]} out of {conf_mat[true_cls].sum()} samples")
print()
print("TODO (Exercise 3): In your notes, answer —")
print("  1. Does this confusion make physical/visual sense?")
print("  2. What feature would distinguish these classes?")
print("  3. Why can a linear model NOT extract that feature from raw pixels?")

---
## §15. Summary

| Concept | Key Idea |
|---------|----------|
| **Linear model** | $\hat{y} = Xw + b$ — simplest parameterized model |
| **MSE at zero params** | $E[y^2] = \text{Var}(y) + \text{mean}(y)^2$ — **not** just Var(y) (§4 error fix #17) |
| **Normal equation** | Closed-form solution; use `jnp.linalg.solve`, not `jnp.linalg.inv` |
| **Weight decay** | L2 penalty $\lambda\|w\|^2$ ≡ Gaussian prior on $w$ (MAP); physics: elastic restoring force |
| **Entropy** | $H(p) = -\sum p_k \log p_k$; Gibbs entropy $S = -k_B \sum p_i \ln p_i$ |
| **Cross-entropy** | $H(p,\hat{p}) = -\sum p_k \log \hat{p}_k$; minimizing CE = MLE |
| **KL divergence** | Extra surprise from wrong model; $D_{\text{KL}} \propto$ free energy difference |
| **Softmax** | $\hat{y}_k = \exp(o_k)/\sum_j \exp(o_j)$; log-sum-exp trick for stability |
| **Fashion-MNIST** | 70k images, 10 classes; softmax regression ~83–85% accuracy |
| **Confusion matrix** | Reveals which classes are confused; forward-pointer to CNN (L06) |

---
## §16. Take-Home Extensions

These sections are moved from the in-session content to reduce session load.
They will be unified in **Lecture 05 (MLP Deep Dive)**.

### Take-Home A — Polynomial overfitting sweep

Using the `train_with_wd` function from §7, sweep λ over `[0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]`
and plot the final train MSE vs. test MSE as a function of λ.

- Identify the optimal λ
- Describe the bias–variance trade-off in your own words
- Bring your plot to L05 where we will compare it with the MLP overfitting demo

### Take-Home B — Flax NNX reimplementation

Reimplement the Fashion-MNIST softmax regression using **Flax NNX** (`flax.nnx`):
- `nnx.Linear(784, 10, rngs=nnx.Rngs(0))` for the single linear layer
- `nnx.Optimizer` with `optax.sgd(0.1)` for the optimizer
- Verify that your NNX model achieves the same test accuracy as the from-scratch version

> See handson02.ipynb §7 (Cells 20–23) in the original Lecture 04 materials for a complete reference implementation.


---
## What's Next?

**Lecture 05 — MLP Deep Dive: Optimization, Overfitting, Init, Uncertainty**

- Add **hidden layers** to break the linear decision boundary limit we hit with softmax regression
- Single canonical **polynomial overfitting / bias–variance demo** (unifying Take-Home A)
- **Xavier / He initialization** — why random init matters for deep nets
- **MC-Dropout** and **deep ensembles** for uncertainty quantification
- Physics payoff: uncertainty-aware predictions for noisy experimental data

> **Physics forward-pointer**: The Ising phase classification task (L06) and Galaxy10 CNN (L07)
> will show how a CNN reduces the confusion between Pullover/Shirt that the linear model struggles with —
> by learning local texture filters rather than using raw pixel intensities.
